# **Member 2 Research questions**

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

cd_month = pd.read_csv("/content/cd_month.csv")

The two questions are district-level comparisons. Monthly records are therefore aggregated across the full period for each community district.

In [14]:
districts = (
    cd_month.groupby(["community_district", "cd_label", "borough"], as_index=False)
    .agg(
        new_filings=("new_filings", "sum"),
        new_units=("new_units", "sum"),
        ksi=("ksi", "sum"),
        severe_inj=("severe_inj", "sum"),
        crashes=("crashes", "sum"),
        history_known=("history_known", "sum"),
        repeat_location=("repeat_location", "sum")
    )
)

districts["repeat_share"] = (
    districts["repeat_location"] /
    districts["history_known"].replace(0, np.nan)
)

print(f"Community districts included: {len(districts)}")

Community districts included: 59


## Research Question 1

**Do community districts that are adding more new housing have a larger share of their serious crashes at repeat-risk locations?**

For each district, compare the total proposed new housing units with the proportion of crashes at repeat-risk locations.

**repeat_share = repeat_location / history_known**

Districts with fewer than 50 crashes with known location history are excluded to reduce instability in the calculated share. Spearman correlation tests whether districts with more proposed housing units tend to have a higher repeat-risk share.

In [15]:
rq1 = districts[districts["history_known"] >= 50].copy()

rho1, p1 = stats.spearmanr(rq1["new_units"], rq1["repeat_share"])
overall_repeat_share = (
    rq1["repeat_location"].sum() / rq1["history_known"].sum()
)

print(f"Districts included: {len(rq1)}")
print(f"Spearman correlation: {rho1:.3f}")
print(f"P-value: {p1:.4f}")
print(f"Overall repeat-risk share: {overall_repeat_share:.1%}")

Districts included: 59
Spearman correlation: 0.393
P-value: 0.0021
Overall repeat-risk share: 6.9%


In [16]:
fig1 = px.scatter(
    rq1,
    x="new_units",
    y="repeat_share",
    color="borough",
    hover_name="cd_label",
    hover_data=["new_filings", "history_known", "repeat_location"],
    labels={
        "new_units": "Total proposed new housing units",
        "repeat_share": "Share of crashes at repeat-risk locations",
        "borough": "Borough"
    },
    title=f"New housing units and repeat-risk crash share by district (Spearman ρ = {rho1:.2f})"
)

fig1.update_yaxes(tickformat=".0%")
fig1.add_hline(
    y=overall_repeat_share,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"Overall share: {overall_repeat_share:.1%}"
)
fig1.update_layout(template="plotly_white", height=550)
fig1.show()

### RQ1 — Findings

Across **59 community districts**, the relationship between total proposed new housing units and repeat-risk crash share is positive and moderate (Spearman ρ = **0.393**, p = **0.002**). The overall repeat-risk share is approximately **6.9%**.

This indicates that districts with larger proposed housing pipelines tend to have a higher proportion of crashes at locations with a recorded history of repeat serious crashes.

### RQ1 — Limitations

- **Association, not causation:** the result does not show that new housing causes repeat-risk crashes.
- **Proposed units are not completed housing:** the measure does not confirm that housing was built or occupied.
- **District differences:** population, road network, traffic volume, and district size may affect both housing and crash patterns.
- **Small numerators:** repeat-risk crashes may be relatively few in some districts, making their shares less stable.
- **Geographic aggregation:** district-level results cannot identify specific dangerous intersections or streets.

## Research Question 2

**Do community districts with more new housing filings experience higher total numbers of serious crashes (KSI)?**

This question examines whether districts with more housing filings also tend to have a greater serious-crash burden.

For each district, the analysis sums housing filings and KSI across the full study period. Spearman correlation measures the strength and direction of the relationship between the two district-level totals.

In [17]:
rho2, p2 = stats.spearmanr(districts["new_filings"], districts["ksi"])

print(f"Community districts included: {len(districts)}")
print(f"Spearman correlation: {rho2:.3f}")
print(f"P-value: {p2:.4f}")

Community districts included: 59
Spearman correlation: 0.321
P-value: 0.0131


In [18]:
fig2 = px.scatter(
    districts,
    x="new_filings",
    y="ksi",
    color="borough",
    hover_name="cd_label",
    hover_data=["new_units", "crashes", "severe_inj"],
    labels={
        "new_filings": "Total new housing filings",
        "ksi": "Total killed or seriously injured (KSI)",
        "borough": "Borough"
    },
    title=f"New housing filings and serious crashes by district (Spearman ρ = {rho2:.2f})"
)

trend = np.polyfit(districts["new_filings"], districts["ksi"], 1)
x_line = np.linspace(districts["new_filings"].min(), districts["new_filings"].max(), 100)
fig2.add_trace(go.Scatter(
    x=x_line,
    y=trend[0] * x_line + trend[1],
    mode="lines",
    name="Linear trend",
    line=dict(color="black", dash="dash")
))
fig2.update_layout(template="plotly_white", height=550)
fig2.show()

### RQ2 — Findings

Across **59 community districts**, the relationship between the total number of new housing filings and total KSI is positive and moderate (Spearman ρ = **0.321**, p = **0.013**).

This suggests that districts with more new housing filings tend to have higher total numbers of people killed or seriously injured in traffic crashes.

### RQ2 — Limitations

- **Association, not causation:** more housing filings do not necessarily cause more serious crashes.
- **District size and exposure:** larger or busier districts may naturally have more housing filings and more crashes. The analysis does not adjust for population, traffic volume, or road length.
- **Aggregated totals:** summing across the full period hides changes over time and may be influenced by unusual years or months.

## Overall Conclusion

- **RQ1:** districts with more proposed new housing units tend to have a larger share of crashes at repeat-risk locations.
- **RQ2:** districts with more new housing filings tend to have higher total KSI counts.

Both questions show positive district-level associations, but neither establishes that housing development causes serious crashes. District size and traffic exposure are important limitations to consider when interpreting the findings.